# Day 14 · Pandas 进阶

> **前置**:Day13 已掌握 DataFrame 创建/查看/loc/iloc
> **关键问题**:如何从 10 万行中筛选出"年龄>30 且 城市=北京"? 如何按城市分组计算平均工资? 多表如何像 SQL 一样 join?

**引入:从"看数"到"问数"**

抽问:`df.loc[1:3]` 和 `df.iloc[1:3]` 取的行数一样吗?(不一样, loc 含右端取 3 行, iloc 不含右端取 2 行)上一节学会了"看"数据到某行某列 —— 这一节学会"问"数据,让数据回答你的业务问题。

---

**第 1 讲:布尔索引过滤 + 排序**

**布尔索引过滤** 是用条件表达式生成 True/False 掩码,再据此选行:单条件 `df[df["成绩"] >= 80]`;多条件 `&` 表示 "且",`|` 表示 "或",**每个条件必须用括号**。**query()** 写字符串条件更简洁,用 `@变量` 引用外部变量。

**排序** 用 `sort_values` 按某列值重排: `ascending=True` 升序(默认),`False` 降序;多列排序传列表;排序不改变原 DataFrame,返回新 DataFrame(需赋值)。


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "班级": ["A", "A", "B", "B", "A"],
    "成绩": [85, 92, 78, 88, 65],
    "性别": ["男", "女", "男", "女", "男"]
})

# 单条件:成绩 >= 80
print(df[df["成绩"] >= 80])

# 多条件:且(&),每个条件必须加括号
# 不加括号会报错:(df["成绩"] >= 80) & (df["班级"] == "A")
print(df[(df["成绩"] >= 80) & (df["班级"] == "A")])

# 多条件:或(|)
print(df[(df["成绩"] >= 90) | (df["班级"] == "B")])

# query():字符串表达式,更简洁
print(df.query("成绩 >= 80 and 班级 == 'A'"))

# query() 用 @ 引用外部变量
min_score = 80
print(df.query("成绩 > @min_score"))

# 排序:单字段降序
print(df.sort_values("成绩", ascending=False))

# 排序:多字段不同升降序(先按班级升序,班级相同按成绩降序)
print(df.sort_values(["班级", "成绩"],
                     ascending=[True, False]))


---

**第 2 讲:分组聚合 groupby + agg**

**核心思想**: split → apply → combine(分组 → 对每组做运算 → 合并结果)。`df.groupby("列")["值列"].mean()` 分组后取某列做聚合。**建议先选列再聚合**: `groupby("班级")["成绩"].mean()` 比 `groupby("班级").mean()` 更精准,避免对非数值列误操作。

**agg()** 可以同时定制多种聚合方式:列表形式 `agg(["mean","max","min","count"])` 自动生成列名;**命名聚合语法** `agg(新名=("列","函数"))` 自定义输出列名,推荐在报告中使用。


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "班级": ["A", "A", "A", "B", "B", "B"],
    "性别": ["男", "女", "男", "女", "男", "女"],
    "成绩": [85, 92, 78, 88, 76, 95]
})

# 基本:按班级分组取平均成绩(推荐先选列再聚合)
print(df.groupby("班级")["成绩"].mean())

# 多列分组:班级+性别
print(df.groupby(["班级", "性别"])["成绩"].mean())

# agg + 列表:一次算多个指标
print(df.groupby("班级")["成绩"].agg(
    ["mean", "max", "min", "count"]
))

# 命名聚合语法:输出列名自定义
# 格式:新列名=("原列名", "聚合函数名")
print(df.groupby("班级").agg(
    平均=("成绩", "mean"),
    最高=("成绩", "max"),
    人数=("成绩", "count")
))


---

**第 3 讲:merge 合并 + concat 拼接**

**merge** 像 SQL 的 join,根据关联字段把左右两表拼起来:`how="inner"` 只留两边都有的键(默认);`how="left"` 以左表为主,右表匹配不到填 NaN;`how="right"` 以右表为主;`how="outer"` 两边键的并集。**merge 可能改变行数** —— 合并后务必检查 `.shape`。

**concat** 沿轴拼接,不依赖键:`axis=0`(默认)纵向堆叠行,建议加 `ignore_index=True` 重置索引;`axis=1` 横向拼列。


In [ ]:
import pandas as pd

# 两张表通过 学号 关联
df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})
df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "数学": [90, 85, 78, 88]
})

# inner join(默认):只留两边都有的学号
m_inner = pd.merge(df_stu, df_score, on="学号")
print(m_inner)

# left join:以左表为主,右表不到填 NaN
m_left = pd.merge(df_stu, df_score, on="学号", how="left")
print(m_left)

# outer join:两边键的并集
m_outer = pd.merge(df_stu, df_score, on="学号", how="outer")
print(m_outer)

# concat 纵向堆叠(ignore_index 重置索引)
df1 = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df2 = pd.DataFrame({"A": [7, 8], "B": [9, 10]})
c_v = pd.concat([df1, df2], ignore_index=True)
print(c_v)

# concat 横向拼列(axis=1)
c_h = pd.concat([df1, df2], axis=1)
print(c_h)


**pivot_table 透视表 + 缺失值处理 fillna / dropna**

透视表把长表变宽交叉表:`index` 行分组键,`columns` 列分组键,`values` 要聚合的值列,`aggfunc` 聚合函数,`margins=True` 增加行列合计。

缺失值处理三步走:**发现** `df.isnull().sum()` 按列统计缺失数目;**删除** `df.dropna()` 丢掉含 NaN 行,可用 `subset` 限定检查列,`thresh` 要求至少 N 个非 NaN;**填充** `df.fillna(常数)` 填固定值,传字典不同列填不同值,`method="ffill"` 用上一行前向填充。 **fillna 默认返回新 DataFrame**,需赋值接收或用 `inplace=True`。


In [ ]:
import pandas as pd
import numpy as np

# pivot_table 透视表
df = pd.DataFrame({
    "日期": ["周一", "周一", "周二", "周二", "周三", "周三"],
    "城市": ["北京", "上海", "北京", "上海", "北京", "上海"],
    "销量": [120, 85, 96, 110, 75, 60]
})
# 日期×城市 看销量
pt = df.pivot_table(
    index="日期", columns="城市",
    values="销量", aggfunc="sum"
)
print(pt)

# 透视表 + 行列合计
pt_m = df.pivot_table(
    index="日期", columns="城市",
    values="销量", aggfunc="sum",
    margins=True
)
print(pt_m)

# 缺失值处理
df2 = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "成绩": [85, np.nan, 78, np.nan],
    "班级": ["A", "A", "B", "B"]
})
# 每列缺失计数(必看第一步)
print(df2.isnull().sum())
# dropna:删掉含 NaN 的行
print(df2.dropna())
# dropna + subset:只看成绩列
print(df2.dropna(subset=["成绩"]))
# fillna:填常数 0
print(df2.fillna(0))
# fillna:字典按列填充(成绩列填该列均值)
print(df2.fillna({"成绩": df2["成绩"].mean()}))


---

**今日小结**

本节围绕 Pandas 数据处理的 7 个核心操作 —— **布尔过滤** → **排序** → **groupby** → **agg** → **merge** → **concat** → **pivot_table + 缺失值处理**,覆盖从"问数"到"清洗数"的完整流程。

**更多练习**

- 当堂练:course/lesson14/in_class/practice01-06.py
- 课后作业:course/lesson14/homework/task01-03.py
